# Phase 2 Bonus: Positional Embedding Ablation (Hindi)
This notebook contains all dependencies inlined, trains a language model from scratch WITHOUT positional embeddings, and runs the full evaluation suite.

In [1]:
!pip install -q sacrebleu rouge_score
import os, json, math, re, string, collections, random
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import IterableDataset, DataLoader
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Using device:", device)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.0/129.0 kB 9.6 MB/s eta 0:00:00
Using device: cuda


In [2]:
from kaggle_secrets import UserSecretsClient
import os

secrets = UserSecretsClient()
GITHUB_TOKEN = secrets.get_secret("GITHUB_TOKEN")
REPO_URL = f"https://{GITHUB_TOKEN}@github.com/CL3-410/individual-project-ManyaJain111.git"

if not os.path.isdir("repo/.git"):
    !git clone -b phase2 {REPO_URL} repo

# ... then keep your existing import os, json, re... code below here

Cloning into 'repo'...
remote: Enumerating objects: 642, done.
remote: Counting objects: 100% (245/245), done.
remote: Compressing objects: 100% (167/167), done.
remote: Total 642 (delta 108), reused 198 (delta 71), pack-reused 397 (from 1)
Receiving objects: 100% (642/642), 31.93 MiB | 22.71 MiB/s, done.
Resolving deltas: 100% (278/278), done.


In [3]:
import sys
sys.path.insert(0, "repo/phase2/src")

from model import GPTLanguageModel
from tokenizer_wrapper import TokenizerWrapper
from dataset import get_dataloader

## 1. Tokenizer

In [4]:
import json, collections, re, string

PUNCT = re.escape(string.punctuation + "।॥‘’“”")
WORD_PAT = re.compile(rf" ?[^{PUNCT}\s]+| ?[{PUNCT}]+", re.UNICODE)

class ByteLevelBPETokenizer:
    def __init__(self, vocab_size=32000):
        self.vocab_size = vocab_size
        self.merges = []  
        self.vocab = {}
        self.byte_to_char = {i: chr(i) for i in range(256)}
        self.char_to_byte = {v: k for k, v in self.byte_to_char.items()}
        self.merge_ranks = {}

    def _word_to_symbols(self, w):
        return [self.byte_to_char[b] for b in w.encode("utf-8")]

    def encode(self, text):
        if not hasattr(self, 'cache'): self.cache = {}
        ids = []
        for w in WORD_PAT.findall(text):
            if w not in self.cache:
                symbols = self._word_to_symbols(w)
                while len(symbols) > 1:
                    pairs = [(symbols[i], symbols[i + 1]) for i in range(len(symbols) - 1)]
                    ranked = [(self.merge_ranks[p], i) for i, p in enumerate(pairs) if p in self.merge_ranks]
                    if not ranked: break
                    _, i = min(ranked)
                    symbols = symbols[:i] + [symbols[i] + symbols[i + 1]] + symbols[i + 2:]
                self.cache[w] = [self.vocab.get(s, 0) for s in symbols]
            ids.extend(self.cache[w])
        return ids

    def decode(self, ids):
        if getattr(self, 'id_to_tok', None) is None:
            self.id_to_tok = {v: k for k, v in self.vocab.items()}
        byte_arr = bytearray()
        for i in ids:
            for ch in self.id_to_tok.get(i, ""):
                try: byte_arr.append(self.char_to_byte[ch])
                except KeyError: byte_arr.extend(ch.encode("utf-8"))
        return bytes(byte_arr).decode("utf-8", errors="replace")

    def load(self, data):
        self.vocab = data["vocab"]
        self.vocab_size = data.get("vocab_size", len(self.vocab))
        self.merges = [((a, b), a + b) for a, b in data["merges"]]
        self.merge_ranks = {pair: i for i, (pair, _) in enumerate(self.merges)}


class UnicodeBPETokenizer:
    def __init__(self, vocab_size=37000):
        self.vocab_size = vocab_size
        self.vocab = {}
        self.merges = []
        self.unk_id = None
        self._merge_ranks = {}
        self._id_to_token = {}

    def _encode_symbols(self, text):
        encoded = []
        for word in WORD_PAT.findall(text):
            symbols = list(word)
            while len(symbols) > 1:
                candidates = [
                    (self._merge_ranks[pair], index)
                    for index, pair in enumerate(zip(symbols, symbols[1:]))
                    if pair in self._merge_ranks
                ]
                if not candidates: break
                _, index = min(candidates)
                symbols[index:index + 2] = [symbols[index] + symbols[index + 1]]
            encoded.extend(symbols)
        return encoded

    def encode(self, text):
        return [self.vocab.get(symbol, self.unk_id) for symbol in self._encode_symbols(text)]

    def decode(self, token_ids):
        return "".join(self._id_to_token.get(token_id, "<UNK>") for token_id in token_ids)

    def load(self, data):
        self.vocab = data["vocab"]
        self.vocab_size = data["vocab_size"]
        self.unk_id = data.get("unk_id", 0)
        self.merges = [((left, right), left + right) for left, right in data["merges"]]
        self._merge_ranks = {pair: rank for rank, (pair, _) in enumerate(self.merges)}
        self._id_to_token = {token_id: token for token, token_id in self.vocab.items()}

class TokenizerWrapper:
    def __init__(self, json_path):
        with open(json_path, "r", encoding="utf-8") as f:
            data = json.load(f)
        
        if data.get("format") == "custom_unicode_bpe_v1":
            self.tokenizer = UnicodeBPETokenizer()
        else:
            self.tokenizer = ByteLevelBPETokenizer()
            
        self.tokenizer.load(data)
        
    def encode(self, text):
        return self.tokenizer.encode(text)
        
    def decode(self, ids):
        return self.tokenizer.decode(ids)
        
    @property
    def vocab_size(self):
        return self.tokenizer.vocab_size

# Test it works on Nepali
tok = TokenizerWrapper("repo/nepali/tokenizer/tokenizer.json")
encoded = tok.encode("तार्किक सम्बन्ध अनुसार, मानौं वीरगन्ज को प्रतिव्यक्ति आय 352 हजार रुपैयाँ छ र भक्तपुर को प्रतिव्यक्ति आय 226 हजार रुपैयाँ छ। बढी प्रतिव्यक्ति आय कसको छ?\\nउत्तर: ")
print("Tokens length:", len(encoded))
print("Decoded string:", repr(tok.decode(encoded)))

Tokens length: 40
Decoded string: 'तार्किक सम्बन्ध अनुसार, मानौं वीरगन्ज को प्रतिव्यक्ति आय 352 हजार रुपैयाँ छ र भक्तपुर को प्रतिव्यक्ति आय 226 हजार रुपैयाँ छ। बढी प्रतिव्यक्ति आय कसको छ?\\nउत्तर:'


In [5]:
import os, json, re, string, glob, tarfile, numpy as np
from multiprocessing import Pool, cpu_count
from huggingface_hub import hf_hub_download
import time

EXTRACT_DIR = '/kaggle/working/lma_extracted/hindi-lma-corpus'
TOK_PATH    = 'repo/phase2/tokenizers/hindi_tokenizer_16000.json'
OUT_DIR     = '/kaggle/working'
CHUNK       = 5000

print("[Dataset] Downloading hindi_submission_phase1.tar.gz...")
tar_path = hf_hub_download(repo_id="manyajain111/hindi-lma-corpus", filename="hindi_submission_phase1.tar.gz", repo_type="dataset")
os.makedirs(EXTRACT_DIR, exist_ok=True)
with tarfile.open(tar_path, "r:gz") as tar:
    tar.extractall(path=EXTRACT_DIR)

PUNCT    = re.escape(string.punctuation + "।॥''\"\"")
WORD_PAT = re.compile(rf" ?[^{PUNCT}\s]+| ?[{PUNCT}]+", re.UNICODE)
_vocab = _merge_ranks = _b2c = _cache = None

def init_worker(tok_path):
    global _vocab, _merge_ranks, _b2c, _cache
    with open(tok_path) as f: data = json.load(f)
    _vocab = data['vocab']
    merges = [((a, b), a + b) for a, b in data['merges']]
    _merge_ranks = {p: i for i, (p, _) in enumerate(merges)}
    _b2c = {i: chr(i) for i in range(256)}
    _cache = {}

def encode_chunk(lines):
    out = []
    for line in lines:
        line = line.strip()
        if not line: continue
        for w in WORD_PAT.findall(line):
            if w not in _cache:
                s = [_b2c[b] for b in w.encode('utf-8')]
                while len(s) > 1:
                    pairs  = [(s[i], s[i+1]) for i in range(len(s)-1)]
                    ranked = [(_merge_ranks[p], i) for i, p in enumerate(pairs) if p in _merge_ranks]
                    if not ranked: break
                    _, i = min(ranked)
                    s = s[:i] + [s[i] + s[i+1]] + s[i+2:]
                _cache[w] = [_vocab.get(x, 0) for x in s]
            out.extend(_cache[w])
    return out

# LOW-RAM GENERATOR
def chunk_generator(fpath, chunk_size):
    with open(fpath, encoding='utf-8', errors='ignore') as f:
        chunk = []
        for line in f:
            chunk.append(line)
            if len(chunk) >= chunk_size:
                yield chunk
                chunk = []
        if chunk:
            yield chunk

def tokenize_to_file(fnames, out_path, max_tok=None):
    fpaths = []
    for f in fnames:
        matches = glob.glob(f'{EXTRACT_DIR}/**/{f}', recursive=True)
        if matches: fpaths.append(matches[0])
    if os.path.exists(out_path): os.remove(out_path)
    
    total_tokens = 0
    t0 = time.time()
    
    with open(out_path, 'wb') as f_out:
        for fpath in fpaths:
            print(f"Reading {os.path.basename(fpath)} (Low RAM mode)...")
            chunks = chunk_generator(fpath, CHUNK)
            
            with Pool(cpu_count(), initializer=init_worker, initargs=(TOK_PATH,)) as pool:
                for i, r in enumerate(pool.imap(encode_chunk, chunks)):
                    arr = np.array(r, dtype=np.uint16)
                    f_out.write(arr.tobytes())
                    total_tokens += len(arr)
                    
                    if i % 50 == 0:
                        tps = total_tokens / max(time.time() - t0, 1)
                        print(f'  processed {i} chunks | {total_tokens:,} tokens | {tps:,.0f} tok/s')
                        
                    if max_tok and total_tokens >= max_tok:
                        pool.terminate()
                        break
            if max_tok and total_tokens >= max_tok: break
    print(f'✅ Saved {total_tokens:,} tokens -> {out_path}')

# Create the training file safely
tokenize_to_file(['train.txt', 'hindi_news_data.txt'], f'{OUT_DIR}/hindi_train.bin', max_tok=520_000_000)

[Dataset] Downloading hindi_submission_phase1.tar.gz...


hindi_submission_phase1.tar.gz:   0%|          | 0.00/1.51G [00:00<?, ?B/s]

/tmp/ipykernel_22/1931099255.py:15: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall(path=EXTRACT_DIR)


Reading train.txt (Low RAM mode)...
  processed 0 chunks | 415,042 tokens | 76,062 tok/s
  processed 50 chunks | 21,942,958 tokens | 603,491 tok/s
  processed 100 chunks | 43,176,736 tokens | 712,205 tok/s
  processed 150 chunks | 64,349,688 tokens | 780,944 tok/s
  processed 200 chunks | 85,616,253 tokens | 825,083 tok/s
  processed 250 chunks | 106,875,825 tokens | 865,132 tok/s
  processed 300 chunks | 128,468,166 tokens | 896,697 tok/s
  processed 350 chunks | 149,577,870 tokens | 924,154 tok/s
  processed 400 chunks | 170,876,318 tokens | 949,937 tok/s
  processed 450 chunks | 192,280,480 tokens | 967,667 tok/s
  processed 500 chunks | 213,788,146 tokens | 983,481 tok/s
  processed 550 chunks | 235,307,368 tokens | 1,001,393 tok/s
  processed 600 chunks | 256,680,672 tokens | 1,015,956 tok/s
  processed 650 chunks | 278,058,886 tokens | 1,029,543 tok/s
  processed 700 chunks | 299,301,472 tokens | 1,041,107 tok/s
  processed 750 chunks | 320,646,059 tokens | 1,052,085 tok/s
  proc

## 2. Model (ABLATED: No Positional Embeddings)

In [6]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

class MultiHeadCausalAttention(nn.Module):
    def __init__(self, d_model, n_heads, context_len, dropout=0.1):
        super().__init__()
        assert d_model % n_heads == 0, "d_model must be divisible by n_heads"
        self.d_model = d_model
        self.n_heads = n_heads
        self.dk = d_model // n_heads
        
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)
        
        self.attn_drop = nn.Dropout(dropout)
        self.resid_drop = nn.Dropout(dropout)
        
        mask = torch.triu(torch.ones(context_len, context_len) * float('-inf'), diagonal=1)
        self.register_buffer("mask", mask)

    def forward(self, x):
        B, T, d_model = x.size()
        
        q = self.W_q(x).view(B, T, self.n_heads, self.dk).transpose(1, 2)
        k = self.W_k(x).view(B, T, self.n_heads, self.dk).transpose(1, 2)
        v = self.W_v(x).view(B, T, self.n_heads, self.dk).transpose(1, 2)
        
        scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(self.dk)
        
        scores = scores + self.mask[:T, :T]
        
        attn_weights = F.softmax(scores, dim=-1)
        attn_weights = self.attn_drop(attn_weights)
        
        context = torch.matmul(attn_weights, v)
        
        context = context.transpose(1, 2).contiguous().view(B, T, d_model)
        
        out = self.resid_drop(self.W_o(context))
        return out, attn_weights

class FeedForward(nn.Module):
    def __init__(self, d_model, d_ff, dropout=0.1):
        super().__init__()
        self.linear1 = nn.Linear(d_model, d_ff)
        self.linear2 = nn.Linear(d_ff, d_model)
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, x):
        x = F.gelu(self.linear1(x))
        x = self.linear2(x)
        return self.dropout(x)

class TransformerBlock(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, context_len, dropout=0.1):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.attn = MultiHeadCausalAttention(d_model, n_heads, context_len, dropout)
        self.ln2 = nn.LayerNorm(d_model)
        self.ffn = FeedForward(d_model, d_ff, dropout)
        
    def forward(self, x):
        attn_out, attn_weights = self.attn(self.ln1(x))
        x = x + attn_out
        
        x = x + self.ffn(self.ln2(x))
        return x, attn_weights

class GPTLanguageModel(nn.Module):
    def __init__(self, vocab_size, d_model, n_heads, n_layers, d_ff, context_len, dropout=0.1, pad_token_id=0):
        super().__init__()
        self.context_len = context_len
        self.pad_token_id = pad_token_id
        
        self.token_emb = nn.Embedding(vocab_size, d_model, padding_idx=pad_token_id)
        # ABLATION: Removed self.pos_emb
        self.drop = nn.Dropout(dropout)
        
        self.blocks = nn.ModuleList([
            TransformerBlock(d_model, n_heads, d_ff, context_len, dropout)
            for _ in range(n_layers)
        ])
        
        self.ln_f = nn.LayerNorm(d_model)
        
        self.head = nn.Linear(d_model, vocab_size, bias=False)
        
        self.head.weight = self.token_emb.weight
        
        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None, return_attention=False):
        B, T = idx.size()
        assert T <= self.context_len, f"Sequence length {T} exceeds context length {self.context_len}"
        
        pos = torch.arange(0, T, dtype=torch.long, device=idx.device)
        
        tok_emb = self.token_emb(idx)
        # ABLATION: Removed pos_emb calculation
        x = self.drop(tok_emb) # ABLATION: Removed pos_emb addition
        
        attention_maps = []
        for block in self.blocks:
            x, attn_weights = block(x)
            if return_attention:
                attention_maps.append(attn_weights)
                
        x = self.ln_f(x)
        logits = self.head(x)
        
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1), ignore_index=self.pad_token_id)
            
        if return_attention:
            return logits, loss, attention_maps
        return logits, loss

    @torch.no_grad()
    def greedy_generate(self, idx, max_new_tokens):
        """
        Greedy decoding: at each step, pick the single most probable next token (argmax).
        Deterministic — temperature has no effect.
        Required by Section 2.3 of the spec alongside temperature sampling.
        """
        for _ in range(max_new_tokens):
            idx_cond = idx if idx.size(1) <= self.context_len else idx[:, -self.context_len:]
            logits, _ = self(idx_cond)
            idx_next = logits[:, -1, :].argmax(dim=-1, keepdim=True)
            idx = torch.cat((idx, idx_next), dim=1)
        return idx

    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=1.0, top_k=None):
        """
        Temperature sampling: higher temperature -> more diverse / random output.
        temperature=0.5  -> sharper distribution (more conservative)
        temperature=1.0  -> unmodified distribution
        temperature=1.5  -> flatter distribution (more creative / random)
        """
        for _ in range(max_new_tokens):
            idx_cond = idx if idx.size(1) <= self.context_len else idx[:, -self.context_len:]
            
            logits, _ = self(idx_cond)
            
            logits = logits[:, -1, :] / temperature
            
            if top_k is not None:
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:, [-1]]] = -float('Inf')
                
            probs = F.softmax(logits, dim=-1)
            
            idx_next = torch.multinomial(probs, num_samples=1)
            
            idx = torch.cat((idx, idx_next), dim=1)
            
        return idx

## 3. Dataset Loader

In [7]:
import os, numpy as np, torch
from torch.utils.data import IterableDataset

BIN_MAP = {
    "manyajain111/hindi-lma-corpus":  {"train":"/kaggle/working/hindi_train.bin","validation":"/kaggle/working/hindi_val.bin","test":"/kaggle/working/hindi_test.bin"},
    "manyajain111/nepali-lma-corpus": {"train":"/kaggle/working/nepali_train.bin","validation":"/kaggle/working/nepali_val.bin","test":"/kaggle/working/nepali_test.bin"},
}

class LMAIterableDataset(IterableDataset):
    def __init__(self, hf_dataset_path, tokenizer, context_len, split="train", max_tokens=None, batch_size=1):
        super().__init__()
        self.context_len = context_len
        self.max_tokens = max_tokens
        path = BIN_MAP[hf_dataset_path][split]
        print(f"[Dataset] Memory-mapping {path}...")
        self.tokens = np.memmap(path, dtype=np.uint16, mode='r')
        print(f"[Dataset] {len(self.tokens):,} tokens ready.")

    def __iter__(self):
        n, i, yielded = len(self.tokens), 0, 0
        while i + self.context_len + 1 <= n:
            chunk = self.tokens[i:i+self.context_len+1]
            x = torch.from_numpy(chunk[:-1].astype(np.int64))
            y = torch.from_numpy(chunk[1:].astype(np.int64))
            yield x, y
            i += self.context_len + 1
            yielded += self.context_len
            if self.max_tokens and yielded >= self.max_tokens:
                return

def get_dataloader(hf_dataset_path, tokenizer, context_len, split="train", batch_size=8, max_tokens=None):
    ds = LMAIterableDataset(hf_dataset_path, tokenizer, context_len, split=split, max_tokens=max_tokens)
    return torch.utils.data.DataLoader(ds, batch_size=batch_size, num_workers=0, pin_memory=False)

## 4. Training Function

In [8]:

def train_ablation(data_dir, out_dir, vocab_size=16240, max_steps=10000, batch_size=32):
    os.makedirs(out_dir, exist_ok=True)
    
    # Load dataset
    train_ds = MemmapDataset(os.path.join(data_dir, "hindi_train.bin"), context_len=256)
    train_loader = DataLoader(train_ds, batch_size=batch_size)
    
    # Init ablated model
    model = GPTLanguageModel(vocab_size=vocab_size, d_model=256, n_heads=8, n_layers=6, d_ff=1024, context_len=256)
    model.to(device)
    
    optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=0.01)
    
    model.train()
    step = 0
    pbar = tqdm(total=max_steps, desc="Training Ablated Model")
    
    for idx, targets in train_loader:
        if step >= max_steps:
            break
            
        idx, targets = idx.to(device), targets.to(device)
        optimizer.zero_grad()
        logits, loss = model(idx, targets=targets)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        
        pbar.update(1)
        if step % 500 == 0:
            pbar.set_postfix({"loss": f"{loss.item():.4f}"})
            
        if step > 0 and step % 10000 == 0:
            torch.save(model.state_dict(), os.path.join(out_dir, f"model_ablated_step_{step}.pt"))
            print(f"Saved intermediate checkpoint at step {step}")
        step += 1
        
    torch.save(model.state_dict(), os.path.join(out_dir, "model_ablated.pt"))
    print("Training complete!")
    return model

## 5. Evaluation Suite

In [9]:
import os
import json
import math
import argparse
import collections
import torch
import torch.nn.functional as F
from tqdm import tqdm
import sacrebleu
from rouge_score import rouge_scorer
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np


def evaluate_ppl_bpb(model, val_loader, device, tokenizer, num_batches=200):
    """Compute cross-entropy loss, perplexity, and bits-per-byte.
    
    BPB = CE_nats / (log(2) * avg_bytes_per_token)
    where avg_bytes_per_token is measured by decoding actual token bytes.
    This makes BPB comparable across tokenizers with different fertility.
    """
    model.eval()
    total_loss_nats = 0.0
    total_tokens = 0
    total_bytes = 0

    val_iter = iter(val_loader)
    with torch.no_grad():
        for i in tqdm(range(num_batches), desc="Evaluating PPL/BPB"):
            try:
                x, y = next(val_iter)
            except StopIteration:
                break

            x, y = x.to(device), y.to(device)
            logits, loss = model(x, targets=y)
            n_toks = y.numel()
            total_loss_nats += loss.item() * n_toks
            total_tokens += n_toks

            flat_ids = y.view(-1).cpu().tolist()
            decoded_text = tokenizer.decode(flat_ids)
            total_bytes += len(decoded_text.encode('utf-8'))

    avg_loss = total_loss_nats / total_tokens if total_tokens > 0 else 0.0
    ppl = math.exp(min(avg_loss, 20))
    avg_bytes_per_token = total_bytes / total_tokens if total_tokens > 0 else 1.0
    bpb = (avg_loss / math.log(2)) / avg_bytes_per_token

    return avg_loss, ppl, bpb, avg_bytes_per_token

def compute_repetition_rate(text, n=4):
    """Fraction of n-grams that are repeated (lower = more diverse)."""
    tokens = text.split()
    if len(tokens) < n:
        return 0.0
    ngrams = [tuple(tokens[i:i+n]) for i in range(len(tokens) - n + 1)]
    repeated = len(ngrams) - len(set(ngrams))
    return repeated / len(ngrams) if ngrams else 0.0


def evaluate_generation(model, tokenizer, device, test_prefixes, references):
    """Generate with greedy decoding and temperatures 0.5 / 1.0 / 1.5.
    Compute BLEU-4, chrF, ROUGE-L, Distinct-1/2, and repetition rate for each.
    Spec requires greedy decoding explicitly alongside the three temperatures.
    """
    model.eval()
    rouge = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=False)

    results = {}

    greedy_generations = []
    for prefix in tqdm(test_prefixes, desc="Greedy Generation"):
        enc = tokenizer.encode(prefix)
        if not enc:
            enc = [1]
        x = torch.tensor([enc], dtype=torch.long).to(device)
        with torch.no_grad():
            gen_idx = model.greedy_generate(x, max_new_tokens=50)
        greedy_generations.append(tokenizer.decode(gen_idx[0].cpu().tolist()))

    bleu_g = sacrebleu.corpus_bleu(greedy_generations, [references]).score
    chrf_g = sacrebleu.corpus_chrf(greedy_generations, [references]).score
    rougeL_g = sum(rouge.score(r, g)['rougeL'].fmeasure
                   for r, g in zip(references, greedy_generations)) / max(len(greedy_generations), 1)
    all_w_g = [w for g in greedy_generations for w in g.split()]
    dist1_g = len(set(all_w_g)) / len(all_w_g) if all_w_g else 0.0
    bg = list(zip(all_w_g, all_w_g[1:]))
    dist2_g = len(set(bg)) / len(bg) if bg else 0.0
    rep_g = sum(compute_repetition_rate(g, 4) for g in greedy_generations) / max(len(greedy_generations), 1)
    results['greedy'] = {
        'BLEU-4': round(bleu_g, 4), 'chrF': round(chrf_g, 4), 'ROUGE-L': round(rougeL_g, 4),
        'Distinct-1': round(dist1_g, 4), 'Distinct-2': round(dist2_g, 4),
        'repetition_rate_4gram': round(rep_g, 4), 'generations': greedy_generations,
    }

    for temp in [0.5, 1.0, 1.5]:
        generations = []
        for prefix in tqdm(test_prefixes, desc=f"Sampling Temp {temp}"):
            enc = tokenizer.encode(prefix)
            if not enc:
                enc = [1]
            x = torch.tensor([enc], dtype=torch.long).to(device)
            with torch.no_grad():
                gen_idx = model.generate(x, max_new_tokens=50, temperature=temp)
            gen_text = tokenizer.decode(gen_idx[0].cpu().tolist())
            generations.append(gen_text)

        bleu = sacrebleu.corpus_bleu(generations, [references]).score
        chrf = sacrebleu.corpus_chrf(generations, [references]).score

        rougeL_scores = [rouge.score(ref, gen)['rougeL'].fmeasure
                         for ref, gen in zip(references, generations)]
        rougeL = sum(rougeL_scores) / len(rougeL_scores) if rougeL_scores else 0.0

        all_words = [w for gen in generations for w in gen.split()]
        total_w = len(all_words)
        dist1 = len(set(all_words)) / total_w if total_w > 0 else 0.0
        bigrams_all = list(zip(all_words, all_words[1:]))
        dist2 = len(set(bigrams_all)) / len(bigrams_all) if bigrams_all else 0.0

        rep_rates = [compute_repetition_rate(gen, n=4) for gen in generations]
        rep_rate = sum(rep_rates) / len(rep_rates) if rep_rates else 0.0

        results[float(temp)] = {
            'BLEU-4': round(bleu, 4),
            'chrF': round(chrf, 4),
            'ROUGE-L': round(rougeL, 4),
            'Distinct-1': round(dist1, 4),
            'Distinct-2': round(dist2, 4),
            'repetition_rate_4gram': round(rep_rate, 4),
            'generations': generations,
        }
    return results

def compute_attention_stats(attention_maps):
    """Compute per-head attention entropy and mean attention distance.
    
    Returns:
        entropy[layer][head]: float
        distance[layer][head]: float  (mean position distance attended to)
    """
    entropy_table = []
    distance_table = []
    for layer_idx, attn in enumerate(attention_maps):
        attn_np = attn[0].detach().cpu().numpy()
        n_heads, T, _ = attn_np.shape
        ent_row, dist_row = [], []
        for h in range(n_heads):
            weights = attn_np[h]
            eps = 1e-9
            ent = -np.sum(weights * np.log(weights + eps), axis=-1).mean()
            ent_row.append(float(ent))
            positions = np.arange(T)
            dists = []
            for q in range(T):
                d = np.sum(np.abs(positions - q) * weights[q])
                dists.append(d)
            dist_row.append(float(np.mean(dists)))
        entropy_table.append(ent_row)
        distance_table.append(dist_row)
    return entropy_table, distance_table


def plot_attention(model, tokenizer, device, text, save_dir):
    """Plot attention heatmaps for early and late layers, all heads.
    Also saves attention entropy and distance summary."""
    model.eval()
    enc = tokenizer.encode(text)
    if not enc:
        print('[plot_attention] Could not encode text, skipping.')
        return
    x = torch.tensor([enc], dtype=torch.long).to(device)
    with torch.no_grad():
        _, _, attention_maps = model(x, return_attention=True)

    tokens = [tokenizer.decode([t]) for t in x[0].tolist()]
    n_layers = len(attention_maps)
    n_heads = attention_maps[0].shape[1]

    layer_indices = sorted(set([0, n_layers // 2, n_layers - 1]))
    for layer_idx in layer_indices:
        attn = attention_maps[layer_idx][0].detach().cpu().numpy()
        cols = min(n_heads, 4)
        fig, axes = plt.subplots(1, cols, figsize=(5 * cols, 5))
        if cols == 1:
            axes = [axes]
        for h in range(cols):
            ax = axes[h]
            sns.heatmap(attn[h], ax=ax, cmap='viridis',
                        xticklabels=tokens, yticklabels=tokens,
                        vmin=0, vmax=attn[h].max())
            ax.set_title(f'Layer {layer_idx} Head {h}')
            ax.set_xlabel('Key position')
            ax.set_ylabel('Query position')
            plt.setp(ax.get_xticklabels(), rotation=45, ha='right', fontsize=7)
            plt.setp(ax.get_yticklabels(), rotation=0, fontsize=7)
        plt.tight_layout()
        plt.savefig(os.path.join(save_dir, f'attn_layer{layer_idx}.png'), dpi=150)
        plt.close()

    entropy_table, distance_table = compute_attention_stats(attention_maps)

    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    sns.heatmap(np.array(entropy_table), ax=axes[0], cmap='magma',
                xticklabels=[f'H{h}' for h in range(n_heads)],
                yticklabels=[f'L{l}' for l in range(n_layers)],
                annot=True, fmt='.2f')
    axes[0].set_title('Attention Entropy (nats)')
    axes[0].set_xlabel('Head'); axes[0].set_ylabel('Layer')

    sns.heatmap(np.array(distance_table), ax=axes[1], cmap='viridis',
                xticklabels=[f'H{h}' for h in range(n_heads)],
                yticklabels=[f'L{l}' for l in range(n_layers)],
                annot=True, fmt='.1f')
    axes[1].set_title('Mean Attention Distance (tokens)')
    axes[1].set_xlabel('Head'); axes[1].set_ylabel('Layer')
    plt.tight_layout()
    plt.savefig(os.path.join(save_dir, 'attn_entropy_distance.png'), dpi=150)
    plt.close()

    return {
        'entropy_per_layer_head': entropy_table,
        'distance_per_layer_head': distance_table,
    }

## 6. Run Everything

In [10]:
import torch
import numpy as np

class MemmapDataset(torch.utils.data.IterableDataset):
    def __init__(self, bin_path, context_len):
        self.data = np.memmap(bin_path, dtype=np.uint16, mode='r')
        self.context_len = context_len

    def __iter__(self):
        n = len(self.data)
        i = 0
        # Iterate over the dataset in chunks of size context_len + 1
        while i + self.context_len + 1 <= n:
            chunk = self.data[i:i+self.context_len+1]
            x = torch.from_numpy(chunk[:-1].astype(np.int64))
            y = torch.from_numpy(chunk[1:].astype(np.int64))
            yield x, y
            i += self.context_len + 1

In [11]:

# UPDATE THIS TO POINT TO YOUR KAGGLE PHASE 2 DATASET DIR
DATA_DIR = "/kaggle/working"
OUT_DIR = "bonus_ablation_output"

# 1. Train the model (10,000 steps is enough to prove it breaks without position!)
if os.path.exists(os.path.join(DATA_DIR, "hindi_train.bin")):
    model = train_ablation(DATA_DIR, OUT_DIR, vocab_size=16240, max_steps=61036)
    
    # 2. Evaluate
    print("Running evaluation suite...")
    # Load tokenizer
    # (Assuming you downloaded tokenizer.json from your Phase 2 HF repo)
    # tokenizer = TokenizerWrapper("checkpoints/pretrained/hindi/tokenizer.json")
    
    # val_ds = MemmapDataset(os.path.join(DATA_DIR, "hindi_val.bin"), context_len=256)
    # val_loader = DataLoader(val_ds, batch_size=32)
    # evaluate_ppl_bpb(model, val_loader, device, tokenizer)
else:
    print(f"ERROR: Could not find training data at {DATA_DIR}. Please update DATA_DIR.")

Training Ablated Model: 100%|██████████| 61036/61036 [3:35:49<00:00,  4.71it/s, loss=4.1700]

Training complete!
Running evaluation suite...
